# Session 2: the numbers behind fields and waves
**A. Real-world problem.** How long should the antenna of an FM radio be, and why does a
wifi router use a stubby antenna a few centimetres long? The answer is a wavelength, and a
wavelength comes from a frequency and the speed of light. Before that, we put numbers on the
electric field of charges: Coulomb's law and superposition.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
K = 8.99e9                  # Coulomb constant, N m^2 / C^2
C = 299792458.0             # speed of light, m/s

## Step 1: Coulomb's law for the field of a point charge
(K is the Coulomb constant.) A point charge q (coulombs) makes an electric field of size  E = K |q| / r^2  (volts per
metre, V/m) at distance r (metres). It points away from a positive charge, towards a
negative one.

In [ ]:
def coulomb_field(q, r):
    """Size of the field (V/m) of a point charge q (C) at distance r (m)."""
    return K * abs(q) / r ** 2


def point_field(q, pos, X, Y):
    """Field vector (Ex, Ey) of one charge q at pos = (x, y) metres, on the grid X, Y."""
    dx, dy = X - pos[0], Y - pos[1]
    r3 = np.maximum(np.hypot(dx, dy), 1e-9) ** 3
    return K * q * dx / r3, K * q * dy / r3

## Step 2: superposition and the dipole
Fields from several charges simply *add* as vectors. A dipole is +q and -q a small distance
d apart. Far away on its axis the two fields almost cancel, leaving  E = 2 K q d / r^3,
which falls faster (1/r^3) than the 1/r^2 of a single charge.

In [ ]:
def dipole_grid(q, d, xs, ys):
    """Dipole field on a grid: +q at (-d/2, 0) and -q at (+d/2, 0). Returns Ex, Ey."""
    X, Y = np.meshgrid(xs, ys)
    e1 = point_field(q, (-d / 2, 0.0), X, Y)
    e2 = point_field(-q, (d / 2, 0.0), X, Y)
    return e1[0] + e2[0], e1[1] + e2[1]


def dipole_axis_exact(q, d, r):
    """Exact field on the axis (x = r > d/2), pointing from + towards -: sum of two Coulomb fields."""
    return K * q * (1 / (r - d / 2) ** 2 - 1 / (r + d / 2) ** 2)


def dipole_axis_far(q, d, r):
    """Far-field formula 2 K q d / r^3 (valid for r much larger than d)."""
    return 2 * K * q * d / r ** 3

## Step 3: the wave equation y = A sin(k x - w t)
A wave of amplitude A travelling in +x:  y(x, t) = A sin(k x - w t). Here k = 2 pi / lambda
is the *wavenumber* (rad/m) and w = 2 pi f the *angular frequency* (rad/s). A crest is where
k x - w t stays constant, so it moves at v = w / k = f lambda. For light and radio v = c.

In [ ]:
def wavenumber(lam):
    return 2 * np.pi / lam


def angular_frequency(f):
    return 2 * np.pi * f


def wave_y(x, t, A, k, w):
    """Displacement of a wave travelling to +x."""
    return A * np.sin(k * np.asarray(x) - w * t)


def phase_speed(w, k):
    return w / k


def wavelength(f, v=C):
    return v / f

## Step 4: antenna length and polarisation
A half-wave dipole antenna is a straight conductor whose length is half a wavelength,
L = lambda / 2 = c / (2 f): the electrons slosh back and forth along it in step with the
wave. (A real wire is cut a few percent shorter because of its thickness; here we use
the ideal value.) The electric field of the wave points along the wire: that direction is
the wave's *polarisation*. A receiver that is tilted by an angle theta from it collects
a fraction cos^2(theta) of the power.

In [ ]:
def half_wave_length(f):
    """Ideal half-wave dipole length in metres for frequency f (Hz)."""
    return C / (2.0 * f)


def received_fraction(theta_deg):
    """Fraction of power received by an antenna (or polariser) tilted by theta from the E field."""
    return float(np.cos(np.radians(theta_deg)) ** 2)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print(f"1 nC at 0.5 m: E = {coulomb_field(1e-9, 0.5):.2f} V/m;  at 1.0 m: {coulomb_field(1e-9, 1.0):.2f} V/m (4 times weaker)")
    for r in (0.5, 1.0, 2.0):
        print(f"dipole q = 1 nC, d = 0.1 m, axis r = {r} m: exact {dipole_axis_exact(1e-9, 0.1, r):.4f} V/m, far formula {dipole_axis_far(1e-9, 0.1, r):.4f} V/m")
    lam = wavelength(2.4e9)
    print(f"wifi 2.4 GHz: lambda = {lam * 100:.2f} cm, k = {wavenumber(lam):.1f} rad/m, w = {angular_frequency(2.4e9):.3e} rad/s, "
          f"w/k = {phase_speed(angular_frequency(2.4e9), wavenumber(lam)):.4e} m/s")
    print(f"wifi half-wave dipole: {half_wave_length(2.4e9) * 100:.2f} cm;  FM 100 MHz: {half_wave_length(100e6):.3f} m")
    print("received fraction at 0, 45, 90 degrees:", [round(received_fraction(a), 3) for a in (0, 45, 90)])

    xs = np.linspace(-1.5, 1.5, 121)
    Ex, Ey = dipole_grid(1e-9, 0.6, xs, xs)
    mag = np.hypot(Ex, Ey)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4.2))
    st = axs[0].streamplot(xs, xs, Ex, Ey, color=np.log10(mag + 1e-3), cmap="viridis", density=1.5, linewidth=0.9)
    axs[0].plot([-0.3], [0], "o", ms=12, color="tab:red")
    axs[0].plot([0.3], [0], "o", ms=12, color="tab:blue")
    fig.colorbar(st.lines, ax=axs[0], label="log10 of |E| (V/m)")
    axs[0].set_xlabel("x (m)")
    axs[0].set_ylabel("y (m)")
    axs[0].set_title("Dipole field on a grid (+1 nC, -1 nC, 0.6 m apart)")
    axs[0].set_aspect("equal")
    r = np.linspace(0.5, 6, 200)
    axs[1].loglog(r, dipole_axis_exact(1e-9, 0.6, r), label="exact (two charges added)")
    axs[1].loglog(r, dipole_axis_far(1e-9, 0.6, r), "--", label="2 K q d / r^3")
    axs[1].loglog(r, coulomb_field(1e-9, r), ":", label="single charge K q / r^2")
    axs[1].set_xlabel("distance r along the axis (m)")
    axs[1].set_ylabel("field E (V/m)")
    axs[1].set_title("A dipole fades faster than one charge")
    axs[1].legend()
    axs[1].grid(alpha=0.3, which="both")
    fig.tight_layout()
    fig.savefig(OUT / "dipole_field.png", dpi=110)

    fig2, axs2 = plt.subplots(1, 2, figsize=(10, 4))
    xw = np.linspace(0, 0.5, 600)
    f = 2.4e9
    T = 1 / f
    for t, col in ((0, "tab:blue"), (T / 4, "tab:orange"), (T / 2, "tab:green")):
        axs2[0].plot(xw * 100, wave_y(xw, t, 1.0, wavenumber(lam), angular_frequency(f)), color=col, label=f"t = {t * 1e12:.0f} ps")
    axs2[0].annotate("", xy=(lam * 100, 1.15), xytext=(0, 1.15), arrowprops=dict(arrowstyle="<->"))
    axs2[0].text(lam * 50, 1.2, f"lambda = {lam * 100:.1f} cm", ha="center")
    axs2[0].set_ylim(-1.2, 1.5)
    axs2[0].set_xlabel("position x (cm)")
    axs2[0].set_ylabel("field y (relative to A)")
    axs2[0].set_title("2.4 GHz wifi wave at three instants")
    axs2[0].legend(loc="lower right", fontsize=7)
    axs2[0].grid(alpha=0.3)
    fr = np.logspace(7, 10, 200)
    axs2[1].loglog(fr / 1e6, half_wave_length(fr), color="black")
    for name, fv in (("FM 100 MHz", 100e6), ("wifi 2.4 GHz", 2.4e9)):
        axs2[1].plot(fv / 1e6, half_wave_length(fv), "o", label=f"{name}: {half_wave_length(fv):.3g} m")
    axs2[1].set_xlabel("frequency (MHz)")
    axs2[1].set_ylabel("half-wave dipole length (m)")
    axs2[1].set_title("Higher frequency, shorter antenna")
    axs2[1].legend()
    axs2[1].grid(alpha=0.3, which="both")
    fig2.tight_layout()
    fig2.savefig(OUT / "wave_antenna.png", dpi=110)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print the dipole field on the axis for d = 0.1 m at r = 0.5, 1 and 2 m (see `dipole_axis_exact`) and compute how far the far-field formula is off at each, in percent.
2. **Stretch.** Write `fm_station_length(f_mhz)` that returns the half-wave length in metres, and print it for 88, 100 and 108 MHz. By how much does the best length change across the band?
3. **Explain.** Explain why a crossed polarisation (theta = 90 degrees) receives nothing, using cos^2(theta), and why a phone still works when you tilt it.

## Assessment
- **A. Conceptual.** Why does the field of a dipole fall off as 1/r^3 while the field of a single charge falls off as 1/r^2? What would happen if the two charges had the same sign?
- **B. Numerical.** A 2 nC charge. Find the field 0.3 m away. Then find the field if the distance is doubled, and sanity-check the ratio.
- **C. Graph and simulation interpretation.** In the log-log graph of dipole field against distance, the curves for the exact dipole and for 2 K q d / r^3 touch at large r but separate at small r. Where do they separate and why? Which line is steeper, the dipole or the single charge?
- **D. Design and optimisation challenge.** Design: choose the length of a half-wave dipole for an FM station at 100 MHz. A rod can be bought only in whole centimetres and your radio case is 1.2 m long. What length do you choose, and what trade-off does a shorter rod cause?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    e = L.coulomb_field(1e-9, 0.5)
    checks.append((f"hand calculation: 1 nC at 0.5 m gives 35.96 V/m (got {e:.2f})", abs(e - 35.96) < 0.01))
    checks.append(("doubling the distance makes the field 4 times weaker", abs(L.coulomb_field(1e-9, 1.0) * 4 - L.coulomb_field(1e-9, 0.5)) < 1e-9))
    checks.append(("field size does not depend on the sign of the charge", L.coulomb_field(-2e-9, 1.0) == L.coulomb_field(2e-9, 1.0)))

    # superposition on a grid: the dipole grid equals the sum of the two single-charge fields
    xs = np.linspace(-1, 1, 21) + 0.013
    X, Y = np.meshgrid(xs, xs)
    a = L.point_field(1e-9, (-0.1, 0.0), X, Y)
    b = L.point_field(-1e-9, (0.1, 0.0), X, Y)
    Ex, Ey = L.dipole_grid(1e-9, 0.2, xs, xs)
    checks.append(("superposition: dipole grid = field of +q plus field of -q", np.allclose(Ex, a[0] + b[0]) and np.allclose(Ey, a[1] + b[1])))
    checks.append(("dipole grid is symmetric: Ex(x, y) = Ex(x, -y) on a symmetric grid",
                   np.allclose(L.dipole_grid(1e-9, 0.2, np.linspace(-1, 1, 21), np.linspace(-1, 1, 21))[0],
                               L.dipole_grid(1e-9, 0.2, np.linspace(-1, 1, 21), np.linspace(-1, 1, 21))[0][::-1, :])))
    ratio = L.dipole_axis_far(1e-9, 0.1, 2.0) / L.dipole_axis_exact(1e-9, 0.1, 2.0)
    checks.append(("far dipole field 2kqd/r^3 agrees with the exact sum to 0.2 percent at r = 20 d", abs(ratio - 1) < 0.002))
    checks.append(("dipole falls as 1/r^3: doubling r divides the field by about 8",
                   abs(L.dipole_axis_exact(1e-9, 0.01, 10.0) / L.dipole_axis_exact(1e-9, 0.01, 20.0) - 8) < 0.01))

    # worked example: 2.4 GHz wifi
    lam = L.wavelength(2.4e9)
    k, w = L.wavenumber(lam), L.angular_frequency(2.4e9)
    checks.append((f"worked example: wifi wavelength is 12.5 cm (got {lam * 100:.2f})", abs(lam * 100 - 12.49) < 0.01))
    checks.append((f"worked example: k = 50.3 rad/m (got {k:.1f}) and w = 1.508e10 rad/s", abs(k - 50.3) < 0.1 and abs(w - 1.508e10) < 1e7))
    checks.append(("wave speed w/k equals c", abs(L.phase_speed(w, k) - L.C) < 1))
    checks.append(("a crest moves at v: y is the same at (x, t) and (x + v dt, t + dt)",
                   abs(L.wave_y(0.03, 1e-10, 1.0, k, w) - L.wave_y(0.03 + L.C * 1e-11, 1e-10 + 1e-11, 1.0, k, w)) < 1e-6))
    checks.append(("the wave repeats after one wavelength", abs(L.wave_y(0.01, 0.0, 2.0, k, w) - L.wave_y(0.01 + lam, 0.0, 2.0, k, w)) < 1e-9))

    # design problem: FM antenna
    checks.append((f"design: half-wave dipole for 100 MHz is 1.5 m (got {L.half_wave_length(100e6):.3f})", abs(L.half_wave_length(100e6) - 1.5) < 0.005))
    checks.append(("design: wifi dipole is 6.2 cm", abs(L.half_wave_length(2.4e9) - 0.0625) < 0.0005))
    checks.append(("antenna length is inversely proportional to frequency", abs(L.half_wave_length(200e6) * 2 - L.half_wave_length(100e6)) < 1e-9))

    checks.append(("polarisation: aligned receiver gets everything, crossed gets nothing, 45 degrees gets half",
                   L.received_fraction(0) == 1.0 and abs(L.received_fraction(90)) < 1e-12 and abs(L.received_fraction(45) - 0.5) < 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
